## This is the code to train the model and acquire influence for Class Imbalance Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 10 features in total. The separation is set to 1.5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 7500, and the test size is 500. The number of features is set to 10. Currently, the number of samples within each class is balanced. This can be changed by modifying the cur_ratio. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, the only thing changing here is how we pre-process the datasets. Basically, the number of samples within each class will be changed based on the choice of cur_ratio. This shall decide the level of imbalance within the train and test sets.** For the other common information, Please refer to the base code for more detailed explanation.

**Guideline**:  
Read in / Construct Datasets -> **Choose the Current Imbalance Ratio** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Imbalance Ratio and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [213]:
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [214]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [215]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [216]:
import random
from keras.optimizers import SGD

In [217]:
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
import seaborn as sns
import matplotlib.pyplot as plt

In [218]:
from sklearn.datasets import make_classification

Train_Size: 1000, 2000, 4000, 8000, 16000  
Feature_Size: 10, 20, 40, 80, 160

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing (Change the balance of each class) -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 7500 train set and 500 test set sample. The current ratio is (5,5), which means the class is balanced right now. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the cur_ratio. In this experiment, all the other things are fixed, but the balance between each class is changing to test how the imbalance of class affect the influence estimation.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. Ratios determine the class imbalance level. Sep to make sure the dataset is distinguishable.

In [219]:
train_pool = 16000
test_size = 500
n_features=10
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]
sep = 1.5

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [220]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           class_sep=sep,
                           random_state=seed)

In [221]:
k= 0

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [222]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features+k)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      -2.042923   2.447225   1.459419  -4.895205  -1.931741   2.439100   
1      -3.021719  -2.189454   1.594643  -0.936749   6.580163  -0.856511   
2       1.744477  -1.747769  -1.803517  -3.028068  -0.517037   3.262006   
3      -0.372644   3.341254   1.849793   0.905480   0.359212   1.766193   
4      -1.972171  -3.053477   1.529830   1.741359   0.858712   1.675452   
...          ...        ...        ...        ...        ...        ...   
16495  -1.318844  -1.342788   0.624967  -0.463707  -0.735329   1.971186   
16496  -3.063709  -0.202935  -0.534888  -6.109804  -0.977436   1.353489   
16497   0.408280  -2.252040   2.995770  -0.442872  -0.904981   2.307891   
16498  -0.117513   3.516412  -3.900624  -2.741435  -4.439399  -0.522876   
16499   3.901795  -1.910619  -0.779033  -0.536239  -1.471359   0.685229   

       feature_7  feature_8  feature_9  feature_10  label     id  
0      -0.694749  -3.070256  -3.

4. **The most important thing in this experiment comes here.** First of all, we choose the exact dataset size during the training by setting value of exact_size. Then, the imbalance level can be set by changing the cur_ratio = ratios[4]. Then, the rest of the code will construct our desired dataset based on the ratio from the data pool. 500 samples will be chosen from them to be the test set. The test set is always set to be balanced so that the overall influence of each training sample will not be biased. 

In [223]:
exact_size = 8000

In [224]:
cur_ratio = ratios[2]
print(cur_ratio)

(5, 5)


In [225]:
major, minor = cur_ratio

In [226]:
df0 = df[df.label == 0]  
df1 = df[df.label == 1] 

In [227]:
t0 = int(exact_size * major / (major + minor))
t1 = exact_size - t0 
print(t0,t1)

4000 4000


In [228]:
s0 = df0.sample(n=t0, random_state=seed)
s1 = df1.sample(n=t1, random_state=seed)

In [229]:
df = pd.concat([s0, s1], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

In [230]:
print(df)
print(df["label"].value_counts())

      feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      0.289038   3.692790  -1.628845  -4.070259  -3.898640  -2.381069   
1     -1.134719  -0.343625   2.188912  -0.346478  -2.168913   2.998409   
2      2.349363   3.052903  -0.036891  -1.810737  -1.761960  -0.716674   
3     -2.517421   0.710340   0.669918  -0.929590  -0.414538   4.305263   
4      1.597508   2.097748   3.208776   2.964748   4.507417   1.312706   
...         ...        ...        ...        ...        ...        ...   
7995   1.076650  -1.254884   0.049430  -0.021436  -2.665772  -0.286804   
7996  -2.559948   0.735459   2.895206   5.413906  -1.586808  -1.590768   
7997  -1.806808  -0.704839   1.698431  -3.602001   0.844647   1.772143   
7998  -1.884069   3.350076   5.046379   3.212790   4.683705   1.530636   
7999  -1.132728   0.957130   3.400784   3.359394   3.380457  -1.807406   

      feature_7  feature_8  feature_9  feature_10  label     id  
0     -1.539977   2.073735   3.260903   -0.13

5. **Another important thing in this experiment comes here.** Here, we shall store the corresponding dataframe of each label and id so that we can map the minority group with their influence. That is to say, we need to know which group does each sample belong to.

In [231]:
id_label_df = df[["id", "label"]].copy()
print(id_label_df)
id_label_df.to_csv("5_5_class_labelIDs.csv",index = False)

         id  label
0      1049      0
1      8125      0
2       241      0
3     11660      0
4     14281      1
...     ...    ...
7995   7462      1
7996   3383      1
7997  12979      0
7998     96      1
7999  11710      1

[8000 rows x 2 columns]


6. Cut the dataset into train set and test set. They have equal number to make sure each contribute equally to the overall influence.

In [232]:
n0 = test_size //2
n1 = test_size - n0

In [233]:
g = df.groupby("label", group_keys=False)
test_df = pd.concat([
    g.get_group(0).sample(n=n0, random_state=42, replace=False),
    g.get_group(1).sample(n=n1, random_state=42, replace=False),
]).sample(frac=1, random_state=42)

train_df = df.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [234]:
print(test_df.groupby("label").get_group(0))
print(test_df["label"].value_counts())

     feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
1     1.450480  -2.016497   3.160825  -0.507879   0.762851   2.059002   
3     0.055396   1.513364   1.777987   3.976750   1.483670  -0.656618   
4     0.674792   2.254483   2.336331  -3.025283  -1.237908  -0.460596   
7    -3.180374  -1.534914   0.455395  -0.989944   1.325141   1.546551   
8     1.349300   2.056482  -3.888892  -3.357247  -2.411799  -1.075569   
..         ...        ...        ...        ...        ...        ...   
492  -0.061795  -0.143813  -0.299029  -1.171912  -5.015868   3.278470   
493  -3.816101  -2.236471  -2.139050  -2.361208  -3.433337   0.722355   
494   2.032367   3.509993  -3.364643  -2.090030  -2.293605  -0.005612   
495  -0.412313  -0.977396   0.936829  -1.232980  -2.371427   1.240822   
499   0.478694  -2.689220   1.762500   1.129599  -5.088474  -2.386799   

     feature_7  feature_8  feature_9  feature_10  label     id  
1    -4.502545   0.246839  -0.127491    0.929493      0   

In [235]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[ 2.8903824e-01  3.6927896e+00 -1.6288446e+00 ...  3.2609029e+00
  -1.3463679e-01  1.0490000e-07]
 [-1.1347193e+00 -3.4362537e-01  2.1889119e+00 ...  2.2644105e+00
   2.7948399e+00  8.1249999e-07]
 [ 2.3493626e+00  3.0529029e+00 -3.6890756e-02 ...  1.2172713e+00
   6.1232668e-01  2.4100000e-08]
 ...
 [-1.8068081e+00 -7.0483923e-01  1.6984305e+00 ...  1.1623163e+00
   3.8293433e-01  1.2979000e-06]
 [-1.8840687e+00  3.3500755e+00  5.0463791e+00 ... -1.9356979e+00
  -2.7082527e-01  9.5999999e-09]
 [-1.1327285e+00  9.5713001e-01  3.4007843e+00 ... -4.3463144e-01
   8.5547608e-01  1.1710000e-06]]


In [236]:
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[ 9.87557352e-01 -2.03810644e+00 -2.91228294e+00 ...  1.89538968e+00
  -3.36546230e+00  1.12470002e-06]
 [ 1.45048010e+00 -2.01649714e+00  3.16082525e+00 ... -1.27490819e-01
   9.29493487e-01  1.55300000e-07]
 [ 7.82497376e-02 -1.94566715e+00 -1.00675607e+00 ... -5.66710889e-01
  -1.18393505e+00  1.34979996e-06]
 ...
 [-9.55501378e-01  3.53005934e+00  1.79996622e+00 ... -2.03569818e+00
  -3.52421212e+00  5.26999997e-07]
 [ 3.43912435e+00 -1.09109074e-01 -1.71157491e+00 ...  3.31679404e-01
   2.39646816e+00  8.91500008e-07]
 [ 4.78694081e-01 -2.68921995e+00  1.76250005e+00 ...  1.32055849e-01
  -3.39765579e-01  1.15360001e-06]]


6.1 (Optional) The hidden code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [237]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

# D = pairwise_distances(X_all) 

In [238]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [239]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

7. Now we have the train_ds and test_ds for training

In [240]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

**One thing to be mentioned here is that some datasets may face high loss when changing the balance of classes. That means a more imbalanced set needs more epochs to reach high accuracy, while a more balanced set can be overfit under such epochs. Therefore, we add one mechanism to check if the model is trained to a satisfactory accuracy. By setting target accuracy, you can make sure everytime, the model is trained to a similar level. The allowance can help avoid some of the training fluctuation.**

In [241]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [242]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
patience  = 10        
min_delta = 1e-4        
target_acc = None #0.90
allowance = 2

best_val_acc = -np.inf
best_epoch   = -1
no_improve   = 0

unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))

for ep in range(epochs):
    hist = model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
    model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
    val_acc = float(hist.history["val_accuracy"][-1])
    
    if (target_acc is not None) and (val_acc >= target_acc) and (allowance <=0):
        print(f"Early stopping: target val_accuracy {target_acc:.4f} reached at epoch {ep+1} (val_acc={val_acc:.4f}).")
        break
    elif (target_acc is not None) and (val_acc >= target_acc) and (allowance >0):
        allowance -= 1

base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

30/30 - 1s - loss: 0.9190 - accuracy: 0.4577 - val_loss: 0.7499 - val_accuracy: 0.4780 - 640ms/epoch - 21ms/step
30/30 - 0s - loss: 0.6799 - accuracy: 0.5653 - val_loss: 0.6067 - val_accuracy: 0.6740 - 127ms/epoch - 4ms/step
30/30 - 0s - loss: 0.5862 - accuracy: 0.6983 - val_loss: 0.5483 - val_accuracy: 0.7640 - 126ms/epoch - 4ms/step
30/30 - 0s - loss: 0.5407 - accuracy: 0.7527 - val_loss: 0.5120 - val_accuracy: 0.7980 - 129ms/epoch - 4ms/step
30/30 - 0s - loss: 0.5085 - accuracy: 0.7836 - val_loss: 0.4823 - val_accuracy: 0.8140 - 126ms/epoch - 4ms/step
30/30 - 0s - loss: 0.4812 - accuracy: 0.8080 - val_loss: 0.4553 - val_accuracy: 0.8380 - 153ms/epoch - 5ms/step
30/30 - 0s - loss: 0.4561 - accuracy: 0.8272 - val_loss: 0.4300 - val_accuracy: 0.8540 - 168ms/epoch - 6ms/step
30/30 - 0s - loss: 0.4325 - accuracy: 0.8424 - val_loss: 0.4062 - val_accuracy: 0.8660 - 148ms/epoch - 5ms/step
30/30 - 0s - loss: 0.4102 - accuracy: 0.8547 - val_loss: 0.3841 - val_accuracy: 0.8760 - 157ms/epoch - 

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [243]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [244]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID     Score
0         1049  0.063698
1         8125  0.072848
2          241  0.063207
3        11660  0.057610
4        14281  0.125109
...        ...       ...
7495      7462  0.121855
7496      3383  0.186712
7497     12979 -0.066860
7498        96  0.051463
7499     11710  0.218872

[7500 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [245]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID     Score
0         1049 -0.000002
1         8125 -0.000003
2          241 -0.000012
3        11660 -0.000003
4        14281  0.000059
...        ...       ...
7495      7462  0.000040
7496      3383  0.000086
7497     12979 -0.000026
7498        96  0.000061
7499     11710  0.000190

[7500 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [246]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [247]:
TracIn_sorted.to_csv("TC_7_3_class.csv",index = False)
df_sorted.to_csv("IF_7_3_class.csv",index = False)

In [248]:
# print(df_sorted)

In [249]:
# train_df[train_df['id'] == 14825]